<a href="https://colab.research.google.com/github/V3GA5/CYB2423_Project/blob/main/Task1_AES_CBC_Encryption_%26_Decryption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AES-256-CBC Encryption and Decryption


---


## Project Overview

This notebook implements symmetric encryption and decryption
of a CSV dataset using the AES-256 algorithm in Cipher Block
Chaining (CBC) mode.

### Objectives
1. Load the original CSV dataset.
2. Generate and securely manage an AES-256 encryption key.
3. Apply PKCS7 padding to the plaintext.
4. Encrypt the dataset using AES-256-CBC.
5. Save the IV and ciphertext in a binary file.
6. Extract the encrypted components.
7. Decrypt the ciphertext and remove padding.
8. Verify that the original plaintext is fully recovered.

### Encryption Configuration
- Algorithm: AES-256
- Mode: CBC
- Key size: 256 bits (32 bytes)
- IV size: 128 bits (16 bytes)
- Padding: PKCS7
- Programming language: Python
- Environment: Google Colab

### Before Running The Code:
- Import: /content/University_Scholarship_Applications_Dataset.csv
- Import: /content/encrypted_data.bin
- Add the `AES_KEY_B64_New` value into Google Colab Secrets


## 1. Environment Setup


*   Install and import the required Python libraries for AES encryption, padding, file handling and key management



In [ ]:
# Install dependency
!pip install cryptography

# Import libraries
import os
import base64
import pandas as pd

from pathlib import Path
from cryptography.hazmat.primitives.ciphers import (
    Cipher, algorithms, modes
)
from cryptography.hazmat.primitives import padding
from google.colab import userdata

## 2. Load Original CSV Dataset


*   Read the supplied CSV file in binary mode.
*   Binary reading preserves the exact original bytes, including its encoding and newline characters.



In [ ]:
# Read dataset
# Define the dataset path
file_path = Path(
    "/content/University_Scholarship_Applications_Dataset.csv"
)

# Read the CSV as bytes
with open(file_path, "rb") as file:
    plaintext = file.read()

# Display dataset information
print("ORIGINAL DATASET INFORMATION — BEFORE ENCRYPTION")
print("----------------------------")
print("File name:", file_path.name)
print("Original file size:", len(plaintext), "bytes")
print("Data type:", type(plaintext))
print("First 100 bytes:", plaintext[:100])
# Display the original bytes in hexadecimal format
print("Original bytes (hex):", plaintext[:100].hex())





ORIGINAL DATASET INFORMATION — BEFORE ENCRYPTION
----------------------------
File name: University_Scholarship_Applications_Dataset.csv
Original file size: 17071 bytes
Data type: <class 'bytes'>
First 100 bytes: b'\xef\xbb\xbfApplicationID,StudentID,FullName,UniversityEmail,PhoneNumber,DateOfBirth,Emirate,Program,Enrollme'
Original bytes (hex): efbbbf4170706c69636174696f6e49442c53747564656e7449442c46756c6c4e616d652c556e6976657273697479456d61696c2c50686f6e654e756d6265722c446174654f6642697274682c456d69726174652c50726f6772616d2c456e726f6c6c6d65


## 3. AES Key and IV Management

*   Generate an AES-256 key **- RUNS ONLY ONCE! skip this step if key was GIVEN**
*   Retrieve the existing AES-256 key from Google Colab Secrets.
*   Generate a fresh initialization vector (IV) for CBC mode.
*   The encryption key is stored separately from the encrypted file.
*   The IV is not secret and will be stored alongside the ciphertext.



In [ ]:
#SKIP IT IF KEY WAS GIVEN!!

# Generatea New key
new_key = os.urandom(32)
new_key_b64 = base64.b64encode(new_key).decode("utf-8")

print(new_key_b64)

# Copy the displayed value into the 'AES_KEY_B64' Google Colab Secrets

KEY GENERATED

In [ ]:
# Retrieve the AES key from Google Colab Secrets
key_b64 = userdata.get("AES_KEY_B64_New")

# Decode the Base64 key (stripping any whitespaces/newlines first)
key = base64.b64decode(key_b64.strip(), validate=True)

# Verify key length
assert len(key) == 32, "Invalid AES-256 key length"

print("AES key retrieved successfully.")
print("Key length:", len(key), "bytes")

AES key retrieved successfully.
Key length: 32 bytes


In [ ]:
# Generate a fresh 128-bit initialization vector
iv = os.urandom(16)

# Verify IV length
assert len(iv) == 16, "Invalid IV length"

print("Initialization vector generated successfully.")
print("IV length:", len(iv), "bytes")

Initialization vector generated successfully.
IV length: 16 bytes


## 4. Data Padding (PKCS7)


*   AES operates on fixed 128-bit (16-byte) blocks.
*   PKCS7 padding is applied to ensure that the plaintext length is a multiple of the AES block size.



In [ ]:
# Create PKCS7 padding object
padder = padding.PKCS7(128).padder()

# Apply padding to original plaintext
padded_data = (
    padder.update(plaintext) + padder.finalize()
)

# Display padding information
print("PKCS7 PADDING VERIFICATION")
print("--------------------------")
print("Original size:", len(plaintext), "bytes")
print("Padded size:", len(padded_data), "bytes")
print("Padding added:", len(padded_data) - len(plaintext), "bytes")
print("Block alignment:", len(padded_data) % 16 == 0)

PKCS7 PADDING VERIFICATION
--------------------------
Original size: 17071 bytes
Padded size: 17072 bytes
Padding added: 1 bytes
Block alignment: True


## 5. AES-256-CBC Encryption


*   Create an AES cipher using the 256-bit key and 128-bit IV.
*   Initialize the encryption context and encrypt the padded plaintext to produce the ciphertext.



In [ ]:
# Create AES-256-CBC cipher
cipher = Cipher(
    algorithms.AES(key),
    modes.CBC(iv)
)

# Create encryption context
encryptor = cipher.encryptor()

In [ ]:
# Encrypt the padded data
ciphertext = (
    encryptor.update(padded_data)
    + encryptor.finalize()
)

print("AES-256-CBC encryption completed.")
print("Ciphertext size:", len(ciphertext), "bytes")

AES-256-CBC encryption completed.
Ciphertext size: 17072 bytes


## 6. Encryption Verification


*   Verify that encryption produced ciphertext with the expected length and AES block alignment.
*   Confirm that the ciphertext differs from the padded plaintext.



In [ ]:
print("AES-256-CBC ENCRYPTION VERIFICATION")
print("-----------------------------------")

print("Padded data length:", len(padded_data), "bytes")
print("Ciphertext length:", len(ciphertext), "bytes")

print("Ciphertext divisible by 16:",
      len(ciphertext) % 16 == 0)

print("Ciphertext equals plaintext:",
      ciphertext == padded_data)

print("Key length:", len(key), "bytes")
print("IV length:", len(iv), "bytes")

encryption_successful = (
    len(ciphertext) == len(padded_data)
    and ciphertext != padded_data
    and len(ciphertext) % 16 == 0
    and len(key) == 32
    and len(iv) == 16
)

print("Encryption successful:", encryption_successful)

assert encryption_successful, "Encryption verification failed"

AES-256-CBC ENCRYPTION VERIFICATION
-----------------------------------
Padded data length: 17072 bytes
Ciphertext length: 17072 bytes
Ciphertext divisible by 16: True
Ciphertext equals plaintext: False
Key length: 32 bytes
IV length: 16 bytes
Encryption successful: True


## 7. Save Encrypted Binary File




> **RUNS ONLY ONCE! skip this step if 'encrypted_data.bin' was imported**



*   Combine the 16-byte IV and ciphertext into a single binary file.
*   The IV is stored at the beginning of the file,
followed by the ciphertext.



In [ ]:
# Combine IV and ciphertext
encrypted_data = iv + ciphertext

# Define output path
encrypted_file_path = Path("/content/encrypted_data.bin")

# Save encrypted data
with open(encrypted_file_path, "wb") as file:
    file.write(encrypted_data)

print("Encrypted file saved successfully.")
print("Encrypted file size:", len(encrypted_data), "bytes")

Encrypted file saved successfully.
Encrypted file size: 17088 bytes


## 8. Encrypted File Verification


*   Read the saved encrypted binary file.
*   Extract the IV from the first 16 bytes and the ciphertext from the remaining bytes.
*   Compare both extracted components with their original values.


In [ ]:
# Read encrypted binary file
with open(encrypted_file_path, "rb") as file:
    encrypted_file = file.read()

# Extract IV and ciphertext
extracted_iv = encrypted_file[:16]
extracted_ciphertext = encrypted_file[16:]

# Verify extracted components
print("ENCRYPTED FILE VERIFICATION")
print("---------------------------")

print("Total file size:", len(encrypted_file), "bytes")
print("Extracted IV length:", len(extracted_iv), "bytes")
print("Extracted ciphertext length:",
      len(extracted_ciphertext), "bytes")

print("IV matches original:", extracted_iv == iv)

print("Ciphertext matches original:",
      extracted_ciphertext == ciphertext)

ENCRYPTED FILE VERIFICATION
---------------------------
Total file size: 17088 bytes
Extracted IV length: 16 bytes
Extracted ciphertext length: 17072 bytes
IV matches original: True
Ciphertext matches original: True


## 9. AES-256-CBC Decryption

*   Recreate the AES-256-CBC cipher using the original encryption key and the extracted IV.
*   Initialize a decryption context and decrypt the ciphertext.
*   The result is the original padded plaintext.

In [ ]:
# Create AES-256-CBC decryption cipher
decrypt_cipher = Cipher(
    algorithms.AES(key),
    modes.CBC(extracted_iv)
)

# Create decryption context
decryptor = decrypt_cipher.decryptor()

In [ ]:
# Decrypt extracted ciphertext
decrypted_padded_data = (
    decryptor.update(extracted_ciphertext)
    + decryptor.finalize()
)

print("Decryption completed.")
print("Decrypted data length:",
      len(decrypted_padded_data), "bytes")

Decryption completed.
Decrypted data length: 17072 bytes


## 10. Decryption Verification

*   Compare the decrypted padded data with the original padded plaintext.
*   A successful comparison confirms that the decryption process recovered the exact padded bytes.




In [ ]:
print("AES-256-CBC DECRYPTION VERIFICATION")
print("-----------------------------------")

print("Decrypted data length:",
      len(decrypted_padded_data), "bytes")

print("Matches original padded data:",
      decrypted_padded_data == padded_data)

assert decrypted_padded_data == padded_data, (
    "Decryption verification failed"
)

AES-256-CBC DECRYPTION VERIFICATION
-----------------------------------
Decrypted data length: 17072 bytes
Matches original padded data: True


## 11. Remove PKCS7 Padding

*   The decrypted data still contains the PKCS7 padding added before encryption.
*   Create an unpadding context and remove the padding to recover the original CSV bytes.



In [ ]:
# Create PKCS7 unpadding object
unpadder = padding.PKCS7(128).unpadder()

# Remove padding
recovered_plaintext = (
    unpadder.update(decrypted_padded_data)
    + unpadder.finalize()
)

print("PKCS7 padding removal completed.")
print("Recovered plaintext length:",
      len(recovered_plaintext), "bytes")

PKCS7 padding removal completed.
Recovered plaintext length: 17071 bytes


## 12. Plaintext Recovery Verification

*   Compare the recovered plaintext against the original CSV bytes.
*   Verify that the recovered data has the same length and is byte-for-byte identical to the original dataset.



In [ ]:
print("PLAINTEXT RECOVERY VERIFICATION")
print("--------------------------------")

print("Original plaintext length:",
      len(plaintext), "bytes")

print("Decrypted padded data length:",
      len(decrypted_padded_data), "bytes")

print("Recovered plaintext length:",
      len(recovered_plaintext), "bytes")

print("Padding successfully removed:",
      len(recovered_plaintext) == len(plaintext))

print("Recovered data matches original:",
      recovered_plaintext == plaintext)

recovery_successful = (
    len(recovered_plaintext) == len(plaintext)
    and recovered_plaintext == plaintext
)

assert recovery_successful, (
    "Original plaintext recovery failed"
)

PLAINTEXT RECOVERY VERIFICATION
--------------------------------
Original plaintext length: 17071 bytes
Decrypted padded data length: 17072 bytes
Recovered plaintext length: 17071 bytes
Padding successfully removed: True
Recovered data matches original: True


## 13. Save Recovered CSV Dataset

*   Save the recovered plaintext bytes as a new CSV file.
*   The recovered file can now be opened and compared
with the original dataset.




In [ ]:
# Define recovered CSV output path
recovered_file_path = Path(
    "/content/Recovered_University_Scholarship_Applications.csv"
)

# Save recovered plaintext
with open(recovered_file_path, "wb") as file:
    file.write(recovered_plaintext)

print("Recovered CSV saved successfully.")
print("Recovered file:", recovered_file_path.name)
print("Recovered file size:",
      recovered_file_path.stat().st_size, "bytes")

Recovered CSV saved successfully.
Recovered file: Recovered_University_Scholarship_Applications.csv
Recovered file size: 17071 bytes


## 14. Final Implementation Summary

*   Summarize the encryption and decryption results.
recovery results.
*   Compare the original, encrypted, and recovered bytes and CSV data.
*   Confirm success when the recovered data matches the original CSV exactly.




In [ ]:

print("=" * 55)
print("       AES-256-CBC IMPLEMENTATION SUMMARY")
print("=" * 55)

results = {
    "Original dataset loaded": len(plaintext) > 0,
    "Original data stored as bytes": isinstance(plaintext, bytes),
    "AES-256 key valid": len(key) == 32,
    "IV valid": len(iv) == 16,
    "PKCS7 padding applied": len(padded_data) % 16 == 0,
    "Encryption successful": encryption_successful,
    "Encrypted data differs from original": ciphertext != plaintext,
    "Encrypted file saved": encrypted_file_path.exists(),
    "Encrypted file components verified": (
        extracted_iv == iv
        and extracted_ciphertext == ciphertext
    ),
    "Decryption successful": decrypted_padded_data == padded_data,
    "Padding removed": len(recovered_plaintext) == len(plaintext),
    "Original plaintext recovered": recovery_successful,
    "Byte-for-byte comparison successful": (
        recovered_plaintext == plaintext
    ),
    "Recovered CSV saved": recovered_file_path.exists()
}

for check, result in results.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("-" * 55)
print(f"Overall result: {'SUCCESS' if all(results.values()) else 'FAILURE'}")
print("=" * 55)


# BYTE COMPARISON TABLE
print("\nBYTE COMPARISON: ORIGINAL vs ENCRYPTED vs RECOVERED")

display_count = min(10, len(plaintext), len(ciphertext),
                    len(recovered_plaintext))

comparison_table = pd.DataFrame({
    "Position": range(display_count),
    "Original (Hex)": [f"{b:02X}" for b in plaintext[:display_count]],
    "Encrypted (Hex)": [f"{b:02X}" for b in ciphertext[:display_count]],
    "Recovered (Hex)": [
        f"{b:02X}" for b in recovered_plaintext[:display_count]
    ],
    "Recovery Match": [
        plaintext[i] == recovered_plaintext[i]
        for i in range(display_count)
    ]
})

display(comparison_table)

print("\nOriginal vs Encrypted:",
      "DIFFERENT" if plaintext != ciphertext else "IDENTICAL")

print("Original vs Recovered:",
      "IDENTICAL" if plaintext == recovered_plaintext else "DIFFERENT")

print("=" * 200)

# FIRST 10 ROWS: ORIGINAL CSV
print("\nFIRST 10 ROWS — ORIGINAL PLAINTEXT")

original_df = pd.read_csv(file_path)
display(original_df.head(10))

print("-" * 200)

# FIRST 10 ROWS: RECOVERED CSV
print("\nFIRST 10 ROWS — RECOVERED PLAINTEXT")

recovered_df = pd.read_csv(recovered_file_path)
display(recovered_df.head(10))

       AES-256-CBC IMPLEMENTATION SUMMARY
Original dataset loaded: PASS
Original data stored as bytes: PASS
AES-256 key valid: PASS
IV valid: PASS
PKCS7 padding applied: PASS
Encryption successful: PASS
Encrypted data differs from original: PASS
Encrypted file saved: PASS
Encrypted file components verified: PASS
Decryption successful: PASS
Padding removed: PASS
Original plaintext recovered: PASS
Byte-for-byte comparison successful: PASS
Recovered CSV saved: PASS
-------------------------------------------------------
Overall result: SUCCESS

BYTE COMPARISON: ORIGINAL vs ENCRYPTED vs RECOVERED


,Position,Original (Hex),Encrypted (Hex),Recovered (Hex),Recovery Match
0,0,EF,AC,EF,True
1,1,BB,9D,BB,True
2,2,BF,CD,BF,True
3,3,41,F0,41,True
4,4,70,0B,70,True
5,5,70,5A,70,True
6,6,6C,82,6C,True
7,7,69,AF,69,True
8,8,63,7C,63,True
9,9,61,D7,61,True



Original vs Encrypted: DIFFERENT
Original vs Recovered: IDENTICAL

FIRST 10 ROWS — ORIGINAL PLAINTEXT


,ApplicationID,StudentID,FullName,UniversityEmail,PhoneNumber,DateOfBirth,Emirate,Program,EnrollmentYear,GPA,HouseholdIncomeAED,RequestedAmountAED,DecisionStatus,ApprovedAmountAED,ReviewerID,SubmissionDate,DecisionDate,ResearchConsent
0,APP-2026-0065,STU-00065,Sara Sami,student0065@students.example.edu,000-000-0065,19/03/2002,Dubai,Business,2024,3.62,55000,13500,APPROVED,12000.0,OFF-02,12/09/2026,15/10/2026,Yes
1,APP-2026-0055,STU-00055,Noor Nasser,student0055@students.example.edu,000-000-0055,04/04/2007,Dubai,Engineering,2026,2.47,32500,11500,DECLINED,0.0,OFF-04,06/09/2026,03/10/2026,Yes
2,APP-2026-0082,STU-00082,Yousef Haddad,student0082@students.example.edu,000-000-0082,04/07/2001,Abu Dhabi,Engineering,2025,3.42,157500,5500,UNDER_REVIEW,NaN,NaN,24/09/2026,NaN,No
3,APP-2026-0051,STU-00051,Rashid Rahman,student0051@students.example.edu,000-000-0051,07/05/2006,Dubai,Computer Science,2026,2.42,102500,17500,APPROVED,14500.0,OFF-04,24/09/2026,12/10/2026,Yes
4,APP-2026-0009,STU-00009,Hassan Fahad,student0009@students.example.edu,000-000-0009,01/07/2005,Fujairah,Engineering,2026,3.59,40000,4500,PENDING,NaN,NaN,14/09/2026,NaN,Yes
5,APP-2026-0067,STU-00067,Tariq Khalil,student0067@students.example.edu,000-000-0067,26/08/2006,Dubai,Health Sciences,2024,3.34,120000,18500,PENDING,NaN,NaN,15/09/2026,NaN,Yes
6,APP-2026-0040,STU-00040,Tariq Al Noor,student0040@students.example.edu,000-000-0040,21/04/2002,Sharjah,Business,2022,3.16,40000,20500,APPROVED,10000.0,OFF-01,09/09/2026,09/10/2026,Yes
7,APP-2026-0047,STU-00047,Hamdan Khalil,student0047@students.example.edu,000-000-0047,15/03/2001,Sharjah,Health Sciences,2024,3.80,80000,16000,APPROVED,11500.0,OFF-04,10/09/2026,05/10/2026,Yes
8,APP-2026-0028,STU-00028,Noor Mansour,student0028@students.example.edu,000-000-0028,15/03/2001,Sharjah,Computer Science,2022,2.41,132500,24000,UNDER_REVIEW,NaN,NaN,16/09/2026,NaN,No
9,APP-2026-0012,STU-00012,Saeed Aziz,student0012@students.example.edu,000-000-0012,05/03/2002,Fujairah,Engineering,2024,2.94,42500,18500,DECLINED,0.0,OFF-01,12/09/2026,08/10/2026,Yes


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

FIRST 10 ROWS — RECOVERED PLAINTEXT


,ApplicationID,StudentID,FullName,UniversityEmail,PhoneNumber,DateOfBirth,Emirate,Program,EnrollmentYear,GPA,HouseholdIncomeAED,RequestedAmountAED,DecisionStatus,ApprovedAmountAED,ReviewerID,SubmissionDate,DecisionDate,ResearchConsent
0,APP-2026-0065,STU-00065,Sara Sami,student0065@students.example.edu,000-000-0065,19/03/2002,Dubai,Business,2024,3.62,55000,13500,APPROVED,12000.0,OFF-02,12/09/2026,15/10/2026,Yes
1,APP-2026-0055,STU-00055,Noor Nasser,student0055@students.example.edu,000-000-0055,04/04/2007,Dubai,Engineering,2026,2.47,32500,11500,DECLINED,0.0,OFF-04,06/09/2026,03/10/2026,Yes
2,APP-2026-0082,STU-00082,Yousef Haddad,student0082@students.example.edu,000-000-0082,04/07/2001,Abu Dhabi,Engineering,2025,3.42,157500,5500,UNDER_REVIEW,NaN,NaN,24/09/2026,NaN,No
3,APP-2026-0051,STU-00051,Rashid Rahman,student0051@students.example.edu,000-000-0051,07/05/2006,Dubai,Computer Science,2026,2.42,102500,17500,APPROVED,14500.0,OFF-04,24/09/2026,12/10/2026,Yes
4,APP-2026-0009,STU-00009,Hassan Fahad,student0009@students.example.edu,000-000-0009,01/07/2005,Fujairah,Engineering,2026,3.59,40000,4500,PENDING,NaN,NaN,14/09/2026,NaN,Yes
5,APP-2026-0067,STU-00067,Tariq Khalil,student0067@students.example.edu,000-000-0067,26/08/2006,Dubai,Health Sciences,2024,3.34,120000,18500,PENDING,NaN,NaN,15/09/2026,NaN,Yes
6,APP-2026-0040,STU-00040,Tariq Al Noor,student0040@students.example.edu,000-000-0040,21/04/2002,Sharjah,Business,2022,3.16,40000,20500,APPROVED,10000.0,OFF-01,09/09/2026,09/10/2026,Yes
7,APP-2026-0047,STU-00047,Hamdan Khalil,student0047@students.example.edu,000-000-0047,15/03/2001,Sharjah,Health Sciences,2024,3.80,80000,16000,APPROVED,11500.0,OFF-04,10/09/2026,05/10/2026,Yes
8,APP-2026-0028,STU-00028,Noor Mansour,student0028@students.example.edu,000-000-0028,15/03/2001,Sharjah,Computer Science,2022,2.41,132500,24000,UNDER_REVIEW,NaN,NaN,16/09/2026,NaN,No
9,APP-2026-0012,STU-00012,Saeed Aziz,student0012@students.example.edu,000-000-0012,05/03/2002,Fujairah,Engineering,2024,2.94,42500,18500,DECLINED,0.0,OFF-01,12/09/2026,08/10/2026,Yes
